<details><summary style="display:list-item; font-size:16px; color:blue;">Jupyter Help</summary>
    
Having trouble testing your work? Double-check that you have followed the steps below to write, run, save, and test your code!

[Click here for a walkthrough GIF of the steps below](https://static-assets.codecademy.com/Courses/ds-python/jupyter-help.gif)

Run all initial cells to import libraries and datasets. Then follow these steps for each question:

1. Add your solution to the cell with `## YOUR SOLUTION HERE ##`.
2. Run the cell by selecting the `Run` button or the `Shift`+`Enter` keys.
3. Save your work by selecting the `Save` button, the `command`+`s` keys (Mac), or `control`+`s` keys (Windows).
4. Select the `Test Work` button at the bottom left to test your work.

![Screenshot of the buttons at the top of a Jupyter Notebook. The Run and Save buttons are highlighted](https://static-assets.codecademy.com/Paths/ds-python/jupyter-buttons.png)

#### Setup Cell

Run the setup cell below to import the libraries required to build agents with tools in LangChain.

- `AgentExecutor` runs the agent loop, managing tool calls and responses. 
- `create_tool_calling_agent` builds an agent that decides when and how to use tools. 
- `tool converts` Python functions into LangChain tools. 
- `ChatOpenAI` provides access to an OpenAI chat model. 
- `DuckDuckGoSearchRun` enables web search as a built-in tool. 

The language model `llm` is initialized with `gpt-4o` at zero temperature for consistent, deterministic responses. 

**Note:** The OpenAI API key is already configured in this workspace, so no additional setup is needed. When working in an external environment, refer to the [OpenAI API](https://platform.openai.com/docs/quickstart?language=python) documentation for details on configuring your API key.

In [ ]:
# Core imports for agents and tools
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# For building prompts and chains
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

# For web search capability
from langchain_community.tools import DuckDuckGoSearchRun

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o", temperature=0, stream=False)

**Note:** Be sure to run the Setup cell above before beginning with the checkpoints!

#### Chains vs. Agents

Before building an agent, let's understand how it differs from a chain. 

A chain follows a fixed sequence: `Input → Prompt → Model → Parser → Output`. The developer defines all steps, and the model cannot decide to do something different. 

An agent is autonomous: it can reason about what to do, decide which tools to use, and repeat steps until the task is complete. 

The example below runs a simple chain. The chain passes the question through the prompt, the language model, and `StrOutputParser` in a fixed sequence. It cannot search the web or call external tools.

In [ ]:
# Example: A simple chain 
prompt = ChatPromptTemplate.from_template("Answer briefly: {question}")
chain = prompt | llm | StrOutputParser()

result = chain.invoke({"question": "What is Python?"})
print(result)

#### Checkpoint 1/5: Creating a Tool for Addition

Use the `@tool` decorator to create a tool named `add_numbers`. This decorator converts a Python function into a LangChain tool using the function name as the tool name, the docstring as the description, and the type hints as the arguments. 

- Define the function with two `float` parameters: `a` and `b`. 
- Add a docstring that describes what the tool does and when to use it. 
- Return the sum of `a` and `b`. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the add_numbers tool
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together.
    
    Use this tool when you need to calculate the sum of two numbers.
    
    Args:
        a: The first number
        b: The second number
    """
    return a + b

#### Inspecting the `add_numbers` Tool

Let's confirm that the `add_numbers` tool was created correctly by reviewing its properties and testing it with a simple example.

In [ ]:
# View the tool's properties
print(f"Tool Name: {add_numbers.name}")
print(f"Tool Description: {add_numbers.description}")
print(f"Tool Arguments: {add_numbers.args}")

# Test the tool directly
result = add_numbers.invoke({"a": 5, "b": 3})
print(f"\nTest: 5 + 3 = {result}")

#### Checkpoint 2/5: Create a Tool for Web Search

Next, let's create a tool that searches the web for current information. Unlike `add_numbers`, which performs a calculation, `web_search` uses `DuckDuckGoSearchRun` to retrieve real-time data. We will wrap it in a custom tool to add a description and control how the agent uses it. 

- Initialize `DuckDuckGoSearchRun` and store it in a variable named `search_engine` to enable web search functionality. 
- Apply the `@tool` decorator to convert the function into a LangChain tool. 
- Define a function named `web_search` with a single `str` parameter named `query`:  
   - Add a docstring that describes what the tool does and when to use it to find recent news, look up facts, or get current information. 
   - Call `.run()` on `search_engine` and pass `query` as the argument to execute the search. 
   - Return the search result from `search_engine` as the function's output. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Initialize the search engine
search_engine = DuckDuckGoSearchRun()

# Define the web_search tool
@tool
def web_search(query: str) -> str:
    """Search the web for current information.
    
    Use this tool when you need to find recent news, 
    look up facts, or get current information.
    
    Args:
        query: The search terms to look up
    """
    return search_engine.run(query)

#### Inspecting the `web_search` Tool

Now, let’s examine the `web_search` tool by checking its properties and running a quick test query to ensure it works as expected.

In [ ]:
# View the tool's properties
print(f"Tool Name: {web_search.name}")
print(f"Tool Description: {web_search.description}")
print(f"Tool Arguments: {web_search.args}")

# Test the tool directly
result = web_search.invoke({"query": "LangChain framework"})
print(f"\nSearch Result: {result[:100]}...")

#### Checkpoint 3/5: Creating the Agent Prompt

An agent needs a structured prompt to understand its role, receive user input, and track its reasoning steps. Use `ChatPromptTemplate` to create a prompt template named `agent_prompt` with the following messages. 

- A system message that describes the assistant as helpful and clearly instructs it to use available tools when answering questions. 
- A human message containing an `input` placeholder for the user's question. 
- A `MessagesPlaceholder` with `variable_name` set to `"agent_scratchpad"` to store the agent's reasoning steps and tool results. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Create the prompt template
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Use your tools to answer questions."),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

#### Checkpoint 4/5: Building the Agent

Now that the tools and prompt are ready, let's assemble the agent by wiring all components together into a working system. 

- Create a list named `tools` containing `add_numbers` and `web_search` so the agent knows which capabilities are available. 
- Use `create_tool_calling_agent` to create an agent named `agent`, passing in `llm`, `tools`, and `agent_prompt` so the agent knows which model, tools, and prompt to use. 
- Instantiate `AgentExecutor` and store it in a variable named `agent_executor` to manage the agent's run loop:  
   - Pass in `agent` and `tools`. 
   - Set `verbose` to `True` to display the agent's reasoning during execution. 
   - Set `stream_runnable` to `False` to disable streaming. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Define the tools list
tools = [add_numbers, web_search]

# Create the agent
agent = create_tool_calling_agent(llm, tools, agent_prompt)

# Create the executor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    stream_runnable=False
)

#### Checkpoint 5/5: Testing the Agent

With the agent fully assembled, let's test it with two different queries to verify that it selects the correct tool for each task. 

- Invoke `agent_executor` with the input `"What is 42 plus 58?"` and store the result in a variable named `addition_result` to test the `add_numbers` tool. 
- Invoke `agent_executor` with the input `"What is LangChain used for?"` and store the result in a variable named `web_search_result` to test the `web_search` tool. 
- Uncomment the `print()` statements to display the final answers. 

**Note:** Don't forget to run the cell and save the notebook before selecting `Test Work`. Open the `Jupyter Help` toggle at the top of the notebook for more details.

In [ ]:
# Invoke the agent with a math-related question
addition_result = agent_executor.invoke(
    {"input": "What is 42 plus 58?"}
)

# Invoke the agent with a web search query
web_search_result = agent_executor.invoke(
    {"input": "What is LangChain used for?"}
)

print("Addition Result:", addition_result["output"])
print("Web Search Result:", web_search_result["output"])